# OCM Catalyst Yield — Results Walkthrough

**What this notebook is.** A live, verifiable walkthrough of the corrected analysis. Every number
below is computed when you run the cell. Nothing is read from a saved results file.

**Why it exists.** Prof. Taniike (JAIST) reviewed our first report and identified a validation flaw.
This notebook demonstrates the flaw, shows the correction, and reports what survives.

**How to read it.** Each section shows the code first, then the result it produces. Where a function
lives in the shared module `ocm_eval.py`, we print its real source so you can see exactly what ran.

| Section | Question it answers |
|---|---|
| 1 | What is actually in the data? |
| 2 | What was wrong with our original test? |
| 3 | Why did the old result look good? |
| 4 | What does the corrected model achieve? |
| 5 | Does literature data help at all? |
| 6 | Where does the model break, and why? |
| 7 | What catalysts do we recommend making? |
| 8 | Which file produced each number? |

Runtime: about 8 minutes end to end.

## Setup

We import the shared evaluation module. Every experiment in this project uses it, so the notebook and
the committed experiments cannot diverge.

In [1]:
import warnings, json, time
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, inspect
import lightgbm as lgb, xgboost as xgb
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import ocm_eval
from ocm_eval import Data, lgb_params, xgb_params, grouped_folds, row_folds, cat_metrics, TARGET

t0 = time.time()
d = Data.load()
TUNED = json.load(open('grouped_tuning.json'))['confirmation']['tuned']['overrides']
print(f"lab rows        : {len(d.y_lab):,}")
print(f"literature rows : {len(d.y_lit):,}")
print(f"features        : {len(d.features)}")
print(f"tuned overrides : {TUNED}")

lab rows        : 89,074
literature rows : 3,852
features        : 67
tuned overrides : {'n_estimators': 300, 'learning_rate': 0.03, 'num_leaves': 127, 'max_depth': -1, 'min_child_samples': 20, 'subsample': 0.6, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0}


## 1. What is actually in the data

This is the single most important fact in the project. The lab data looks like 89,074 independent
measurements. It is not.

In [2]:
el = [c for c in d.features if c not in ('Temperature_C', 'prep_enc')]

print(f"lab measurements                      : {len(d.y_lab):,}")
print(f"DISTINCT CATALYSTS                    : {d.n_cat:,}")
print(f"measurements per catalyst (mean)      : {len(d.y_lab)/d.n_cat:.1f}")
print(f"distinct temperatures                 : {sorted(d.dl_lab.Temperature_C.unique())}")
print(f"preparations used in the lab          : {d.dl_lab.Preparation.unique().tolist()}")

# How many rows share an IDENTICAL feature vector?
full = d.dl_lab[['Preparation','Temperature_C']+el].astype(str).agg('|'.join, axis=1)
vc = full.value_counts()
print(f"\nunique input vectors                  : {len(vc):,}")
print(f"rows per identical input vector (mean) : {vc.mean():.1f}")

# Those rows are NOT replicates. They are a designed grid of reaction conditions that this
# file does not record. The grid size is recoverable from the row counts alone:
cell = d.dl_lab.groupby([d.groups, d.dl_lab.Temperature_C]).size()
per_cat = d.dl_lab.groupby(d.groups).size()
sizes = cell.reset_index(name='n')
exact135 = per_cat[per_cat == 135].index
full_grid = [c for c in exact135 if sorted(sizes[sizes['level_0'] == c].n) == [27]*5]
print(f"\nmost common rows per (catalyst, temperature) : {cell.mode().iloc[0]}")
print(f"largest such cell anywhere                  : {cell.max()}  (= 2 x 27)")
print(f"cells larger than 54                        : {(cell > 54).sum()}")
print(f"catalysts with exactly 135 rows             : {len(exact135)}")
print(f"   ...splitting as exactly (27,27,27,27,27) : {len(full_grid)} of {len(exact135)}")
print("\n-> 5 temperatures x 27 condition settings = 135, which is the number of conditions")
print("   Prof. Taniike states each catalyst is run under. Rows sharing an input vector are")
print("   therefore DIFFERENT REACTION CONDITIONS, not repeats of one measurement.")

# Variance not reachable from the recorded features. Use the pooled within-cell sum of squares:
# an unweighted mean of per-cell variances would weight a 2-row cell like a 27-row cell.
g = d.dl_lab.groupby(full.values)[TARGET]
n_i, v_i = g.size(), g.var()
within_SS = ((n_i - 1) * v_i).fillna(0).sum()
total_SS = ((d.dl_lab[TARGET] - d.dl_lab[TARGET].mean()) ** 2).sum()
print(f"\nyield variance not reachable from these features : {100*within_SS/total_SS:.1f}%")
print(f"best attainable row-level RMSE with these features: {np.sqrt(within_SS/len(d.y_lab)):.3f}")

lab measurements                      : 89,074
DISTINCT CATALYSTS                    : 917
measurements per catalyst (mean)      : 97.1
distinct temperatures                 : [np.float64(700.0), np.float64(750.0), np.float64(800.0), np.float64(850.0), np.float64(900.0)]
preparations used in the lab          : ['Impregnation']



unique input vectors                  : 4,399
rows per identical input vector (mean) : 20.2

most common rows per (catalyst, temperature) : 27
largest such cell anywhere                  : 54  (= 2 x 27)
cells larger than 54                        : 0
catalysts with exactly 135 rows             : 15
   ...splitting as exactly (27,27,27,27,27) : 15 of 15

-> 5 temperatures x 27 condition settings = 135, which is the number of conditions
   Prof. Taniike states each catalyst is run under. Rows sharing an input vector are
   therefore DIFFERENT REACTION CONDITIONS, not repeats of one measurement.

yield variance not reachable from these features : 19.9%
best attainable row-level RMSE with these features: 1.757


**Why this matters.** A random split by *row* puts the same catalyst on both sides. The model is
then asked to predict a catalyst it has already trained on. That measures recall, not discovery.

Note what the last two numbers are and are not. They are not a measurement-noise floor. The
condition settings that drive the within-cell spread are real and reproducible — they are simply
absent from this file. So that share of the variance is unreachable *with these features*, not
irreducible in principle: recovering the condition columns would make most of it learnable. The
RMSE figure also assumes the model already knows each catalyst's own cell means, which requires
having seen that catalyst — so it is not a headroom target for the unseen-catalyst task at all.

## 2. The validation flaw, demonstrated

Here is the exact code that builds the two kinds of split. This is the real source from
`ocm_eval.py` — not a copy.

In [3]:
print(inspect.getsource(ocm_eval.grouped_folds))
print(inspect.getsource(ocm_eval.row_folds))

def grouped_folds(d: Data, seed=0, n_folds=5):
    """All rows of a catalyst in exactly one fold; group->fold assignment shuffled by seed."""
    rng = np.random.default_rng(seed)
    perm = rng.permutation(d.n_cat)
    fold_of_group = np.empty(d.n_cat, dtype=int)
    for k, chunk in enumerate(np.array_split(perm, n_folds)):
        fold_of_group[chunk] = k
    fold = fold_of_group[d.groups]
    return [(np.where(fold != k)[0], np.where(fold == k)[0]) for k in range(n_folds)]

def row_folds(d: Data, seed=0, n_folds=5):
    """Row-level split — HISTORICAL COMPARISON ONLY. Leaks catalyst identity across folds."""
    return list(KFold(n_folds, shuffle=True, random_state=seed).split(d.X_lab))



Now we prove the difference is real. We count how many catalysts appear on **both** sides of a
split under each method.

In [4]:
for name, mk in [('row-level split', row_folds), ('catalyst-grouped split', grouped_folds)]:
    folds = mk(d, seed=0)
    leaked = 0
    for tr, va in folds:
        leaked += len(set(d.groups[tr]) & set(d.groups[va]))
    print(f"{name:24s}: {leaked:5d} catalyst-fold overlaps across 5 folds")
print("\n-> Under a row split, nearly every catalyst is in both train and test.")
print("   Under a grouped split, none are.")

row-level split         :  4522 catalyst-fold overlaps across 5 folds
catalyst-grouped split  :     0 catalyst-fold overlaps across 5 folds

-> Under a row split, nearly every catalyst is in both train and test.
   Under a grouped split, none are.


### The effect on the headline result

We now run the baseline and our PFT method under both protocols, with the same 5 seeds used in the
committed experiment. This takes a few minutes.

In [5]:
SEEDS = [0, 1, 2, 7, 13]

def run(protocol, model, seeds=SEEDS, **kw):
    mk = grouped_folds if protocol == 'grouped' else row_folds
    out = [ocm_eval.run_cv(d, mk(d, seed=s), s, model, **kw) for s in seeds]
    return np.mean([r['rmse_foldmean'] for r in out]), np.std([r['rmse_foldmean'] for r in out], ddof=1), out

res = {}
for proto in ['row', 'grouped']:
    res[(proto,'baseline')] = run(proto, 'baseline')
    res[(proto,'pft')]      = run(proto, 'pft', s1_kind='qn_joint', use_filter=True)
    print(f"[{time.time()-t0:6.1f}s] {proto} done")

print(f"\n{'protocol':10s} {'baseline':>18s} {'PFT':>18s} {'change':>10s}")
for proto in ['row', 'grouped']:
    b, bs, _ = res[(proto,'baseline')]; p, ps, _ = res[(proto,'pft')]
    print(f"{proto:10s} {b:8.3f} +/- {bs:.3f} {p:8.3f} +/- {ps:.3f} {100*(p-b)/b:+9.1f}%")

[ 188.6s] row done


[ 350.3s] grouped done

protocol             baseline                PFT     change
row           2.118 +/- 0.004    1.911 +/- 0.001      -9.8%
grouped       2.943 +/- 0.031    2.986 +/- 0.032      +1.5%


**Read the two rows against each other.**

- Under the row split, PFT looks about 10% better. This is what we originally reported.
- Under the catalyst split, PFT is slightly *worse* than the baseline.

Note the baseline degrades too (2.12 to 2.94). Roughly half of what looked like skill was recall.
This was never a PFT-specific problem. It was a protocol problem.

## 3. Why the old result looked good

Our Stage-1 model trained on literature data **plus the lab training rows**. Under a row split those
rows contain the test catalysts. So the prior feature carried memorised answers.

Here is the code path. Note that `*_joint` kinds append `Xlab_tr` to the Stage-1 training set.

In [6]:
print(inspect.getsource(ocm_eval.stage1_data))

def stage1_data(kind, Xlit_f, ylit_f, Xlab_tr, ylab_tr):
    """Stage-1 design matrix + labels. WARNING: *_joint kinds include lab training rows in Stage 1 —
    under row-level splits this is the catalyst-identity leakage channel; use only with grouped folds
    or for historical comparison."""
    if kind == 'qn_joint':
        return np.vstack([Xlit_f, Xlab_tr]), np.concatenate([quantile_normalize_y(ylit_f, ylab_tr), ylab_tr])
    if kind == 'raw_joint':
        return np.vstack([Xlit_f, Xlab_tr]), np.concatenate([ylit_f, ylab_tr])
    if kind == 'qn_litonly':
        return Xlit_f, quantile_normalize_y(ylit_f, ylab_tr)
    if kind == 'rank_litonly':
        return Xlit_f, rankdata(ylit_f, method='average') / (len(ylit_f) + 1)
    raise ValueError(kind)



We test it directly. If the gain came from the lab rows, removing them should destroy it.

In [7]:
S3 = [0, 1, 2]
b_row  = run('row', 'baseline', seeds=S3)[0]
b_grp  = run('grouped', 'baseline', seeds=S3)[0]

print(f"{'Stage-1 training set':32s} {'row-level':>12s} {'grouped':>12s}")
for kind, label in [('qn_joint', 'literature + LAB TRAIN ROWS'), ('qn_litonly', 'literature only')]:
    r = run('row', 'pft', seeds=S3, s1_kind=kind, use_filter=True)[0]
    gp = run('grouped', 'pft', seeds=S3, s1_kind=kind, use_filter=True)[0]
    print(f"{label:32s} {r:8.3f} ({100*(r-b_row)/b_row:+5.1f}%) {gp:8.3f} ({100*(gp-b_grp)/b_grp:+5.1f}%)")
print(f"{'baseline (reference)':32s} {b_row:8.3f}          {b_grp:8.3f}")
print("\n-> Removing the lab rows collapses the row-level gain from about -10% to about -2.5%.")
print("   Under the grouped split, neither version beats the baseline.")

Stage-1 training set                row-level      grouped


literature + LAB TRAIN ROWS         1.911 ( -9.8%)    2.977 ( +1.7%)


literature only                     2.068 ( -2.4%)    2.935 ( +0.2%)
baseline (reference)                2.119             2.928

-> Removing the lab rows collapses the row-level gain from about -10% to about -2.5%.
   Under the grouped split, neither version beats the baseline.


## 4. The corrected model

Prof. Taniike pointed out that the practical goal is not to predict the best *condition*. His lab
tests every catalyst across a fixed battery. The goal is to know whether a catalyst can reach a high
yield at all.

So we predict each catalyst's **maximum** yield from composition alone. That is 917 training rows
instead of 89,074.

In [8]:
grp = d.dl_lab.groupby(d.groups)
el_cols = [c for c in d.features if c != 'Temperature_C']
cat_df = grp.agg(**{c: (c, 'first') for c in el_cols}, y_max=(TARGET, 'max'))
Xc = cat_df[el_cols].values.astype(float)
yc = cat_df['y_max'].values
print(f"catalyst-level training table: {Xc.shape[0]} catalysts x {Xc.shape[1]} features")

def fold_assignment(seed, n=len(yc), k=5):
    r = np.random.default_rng(seed); perm = r.permutation(n); f = np.empty(n, int)
    for i, ch in enumerate(np.array_split(perm, k)): f[ch] = i
    return f

def eval_catalyst_model(seed, extra=None):
    """Train on 4/5 of the catalysts, predict the held-out 1/5. Returns pooled predictions."""
    f = fold_assignment(seed); yp = np.empty(len(yc))
    for k in range(5):
        tr, va = np.where(f != k)[0], np.where(f == k)[0]
        sc = StandardScaler().fit(Xc[tr])
        X = np.hstack([sc.transform(Xc), extra]) if extra is not None else sc.transform(Xc)
        m = lgb.LGBMRegressor(**lgb_params(seed, **TUNED)).fit(X[tr], yc[tr])
        yp[va] = m.predict(X[va])
    return yp

runs = [cat_metrics(np.arange(len(yc)), yc, eval_catalyst_model(s)) for s in SEEDS]
print(f"\nUnseen catalysts, {len(SEEDS)} seeds:")
print(f"  Spearman (predicted vs real best yield) : {np.mean([r['spearman_max'] for r in runs]):.3f}")
print(f"  Enrichment of top performers            : {np.mean([r['enrichment_top10pct'] for r in runs]):.2f}x")
print(f"  Hit rate in a 20-catalyst shortlist     : {np.mean([r['precision_at20_vs_top10pct'] for r in runs]):.2f}")

catalyst-level training table: 917 catalysts x 66 features



Unseen catalysts, 5 seeds:
  Spearman (predicted vs real best yield) : 0.761
  Enrichment of top performers            : 4.28x
  Hit rate in a 20-catalyst shortlist     : 0.44


Here is how those metrics are defined. Again, the real source.

In [9]:
print(inspect.getsource(ocm_eval.cat_metrics))

def cat_metrics(cat_codes, yt, yp, top_frac=0.10, budget=20):
    """Catalyst-level screening metrics: max over each catalyst's test rows, predicted vs observed."""
    g = pd.DataFrame({'c': cat_codes, 't': yt, 'p': yp}).groupby('c').agg(t=('t', 'max'), p=('p', 'max'))
    n = len(g); K = max(1, int(round(top_frac * n)))
    top_true = set(g.nlargest(K, 't').index)
    prec = float(np.mean([c in top_true for c in g.nlargest(K, 'p').index]))
    B = min(budget, n)
    prec_budget = float(np.mean([c in top_true for c in g.nlargest(B, 'p').index]))
    return dict(n_catalysts=int(n),
                pearson_max=float(np.corrcoef(g.t, g.p)[0, 1]),
                spearman_max=float(spearmanr(g.t, g.p)[0]),
                precision_top10pct=prec, enrichment_top10pct=float(prec / top_frac),
                precision_at20_vs_top10pct=prec_budget)



**What enrichment means in practice.** An enrichment of about 4x says a synthesis campaign guided
by this model finds top-decile catalysts roughly four times faster than picking at random.

These are point estimates. The honest intervals are wide — we report them in section 8.

### 4b. How much of that score is chemistry, and how much is measurement effort?

The lab did not measure every catalyst equally, and cells that were run further contain better yields.
So part of the score above could be a record of which experiments were carried to completion. Three
checks, computed live below.

In [10]:
n_cat = len(yc)
n_rows_cat = d.dl_lab.groupby(d.groups).size().values
max_cell   = d.dl_lab.groupby([d.groups, d.dl_lab.Temperature_C]).size().groupby(level=0).max().values
EE = max_cell >= 20                      # equal-effort set

def oof_pred(target, seed):
    f = fold_assignment(seed); yp = np.empty(n_cat)
    for k in range(5):
        tr, va = np.where(f != k)[0], np.where(f == k)[0]
        s_ = StandardScaler().fit(Xc[tr])
        m = lgb.LGBMRegressor(**lgb_params(seed, **TUNED)).fit(s_.transform(Xc[tr]), target[tr])
        yp[va] = m.predict(s_.transform(Xc[va]))
    return yp

P = np.mean([oof_pred(yc, s) for s in SEEDS], axis=0)                        # real model
E = np.mean([oof_pred(n_rows_cat.astype(float), s) for s in SEEDS], axis=0)  # effort-only control

def sc(mask, pred):
    m = cat_metrics(np.arange(int(mask.sum())), yc[mask], pred[mask])
    return m['spearman_max'], m['enrichment_top10pct']

ALL = np.ones(n_cat, bool)
print(f"equal-effort set (>=20 rows in >=1 cell): {EE.sum()} of {n_cat} catalysts\n")
for nm, msk, pr in [("REAL model, all catalysts", ALL, P), ("REAL model, equal-effort", EE, P),
                    ("EFFORT-ONLY control, all", ALL, E), ("EFFORT-ONLY control, equal-effort", EE, E)]:
    a, b = sc(msk, pr); print(f"  {nm:36s} spearman {a:.4f}   enrichment {b:.2f}x")

# Is the drop just from scoring fewer catalysts? Random subsets of the SAME predictions.
rng = np.random.default_rng(3)
vals = []
for _ in range(300):
    idx = rng.choice(n_cat, int(EE.sum()), replace=False)
    m = np.zeros(n_cat, bool); m[idx] = True
    vals.append(sc(m, P)[0])
lo, hi = np.percentile(vals, 2.5), np.percentile(vals, 97.5)
print(f"\n  300 RANDOM subsets of size {int(EE.sum())}: spearman {np.mean(vals):.4f}  95% [{lo:.4f}, {hi:.4f}]")
print(f"  equal-effort value {sc(EE,P)[0]:.4f} -> {'BELOW the interval: the drop is REAL' if sc(EE,P)[0] < lo else 'inside: could be a size artifact'}")
print(f"\n  Spearman(measurements, observed max): all {spearmanr(n_rows_cat, yc)[0]:+.4f}"
      f"  ->  equal-effort {spearmanr(n_rows_cat[EE], yc[EE])[0]:+.4f}")

print("\n  Inside the model's own top-K (the regime a campaign lives in):")
for K in [20, 150]:
    t = np.argsort(-P)[:K]
    print(f"    top-{K:<4d} internal spearman {spearmanr(yc[t], P[t])[0]:+.3f}   mean observed max {yc[t].mean():.2f}%")

equal-effort set (>=20 rows in >=1 cell): 771 of 917 catalysts

  REAL model, all catalysts            spearman 0.7672   enrichment 4.35x
  REAL model, equal-effort             spearman 0.7235   enrichment 3.77x
  EFFORT-ONLY control, all             spearman 0.3996   enrichment 0.87x
  EFFORT-ONLY control, equal-effort    spearman 0.2244   enrichment 1.04x



  300 RANDOM subsets of size 771: spearman 0.7671  95% [0.7556, 0.7800]
  equal-effort value 0.7235 -> BELOW the interval: the drop is REAL

  Spearman(measurements, observed max): all +0.2932  ->  equal-effort +0.0029

  Inside the model's own top-K (the regime a campaign lives in):
    top-20   internal spearman -0.066   mean observed max 17.33%
    top-150  internal spearman +0.179   mean observed max 16.71%


**Read those four numbers together.** The equal-effort score is lower, and the random-subset
control shows that is a real coverage effect rather than an artifact of scoring fewer catalysts.

The effort-only control is the sharpest test here. A model trained *only* to predict how many times a
catalyst was measured — it never sees a yield — still reaches a rank correlation around 0.40 against
observed maximum yield. But its enrichment is below 1x, i.e. no better than picking at random. So rank
correlation is partly purchasable from experimental effort; enrichment is not. That is the concrete
reason enrichment, not the correlation, is our primary metric.

The last block is a limit worth stating before any synthesis campaign. Inside the model's own
top-ranked catalysts the internal ordering carries almost no information. The model picks a good
*set* — its top 20 average far above the library mean — but it cannot rank within that set. A
shortlist should be treated as a group to test, not as a league table.

## 5. Does literature data help?

The leak explained the old result. It did not prove literature data is useless. So we tested four
honest ways to use it, with success criteria fixed before running.

In [11]:
# Aggregate literature the same way: one row per unique composition, target = its max yield
elements = [c for c in el_cols if c != 'prep_enc']
lit_id = d.dl_lit[['Preparation'] + elements].astype(str).agg('|'.join, axis=1)
lg = d.dl_lit.groupby(pd.factorize(lit_id)[0])
Xlc = lg[el_cols].first().values.astype(float)
ylc = lg[TARGET].max().values
ylc_rank = rankdata(ylc, method='average') / (len(ylc) + 1)
print(f"unique literature compositions: {len(ylc)}")

from scipy.spatial.distance import cdist

def variant_features(seed, kind):
    """Build the extra feature column(s) each design adds to the composition model."""
    sc = StandardScaler().fit(Xc)
    Xc_sc, Xlc_sc = sc.transform(Xc), sc.transform(Xlc)
    cols = []
    if kind in ('prior', 'gated'):                       # literature expert, rank labels, NO lab rows
        pre = xgb.XGBRegressor(**xgb_params(seed)).fit(Xlc_sc, ylc_rank)
        cols.append(pre.predict(Xc_sc).reshape(-1, 1))
    if kind in ('similarity', 'gated'):                  # how close is this catalyst to the literature
        nn = np.sort(cdist(Xc_sc, Xlc_sc), axis=1)
        cols.append(nn[:, :1]); cols.append(nn[:, :5].mean(1, keepdims=True))
    return np.hstack(cols) if cols else None

out = {}
for kind, label in [(None,'composition only (control)'), ('prior','literature rank prior'),
                    ('similarity','similarity features'), ('gated','prior + similarity')]:
    rr = [cat_metrics(np.arange(len(yc)), yc, eval_catalyst_model(s, variant_features(s, kind) if kind else None))
          for s in SEEDS]
    out[label] = np.mean([r['spearman_max'] for r in rr])

base = out['composition only (control)']
print(f"\n{'design':32s} {'Spearman':>9s} {'vs control':>11s}")
for k, v in out.items():
    print(f"{k:32s} {v:9.3f} {v-base:+11.3f}")
print("\n-> No design beats composition alone.")

unique literature compositions: 1754



design                            Spearman  vs control
composition only (control)           0.761      +0.000
literature rank prior                0.756      -0.005
similarity features                  0.741      -0.019
prior + similarity                   0.744      -0.017

-> No design beats composition alone.


## 6. Where the model breaks, and why

We hold out entire element families. The Ba family is the important case.

In [12]:
has_ba = cat_df['Ba'].values > 0
te, tr = np.where(has_ba)[0], np.where(~has_ba)[0]
top10 = yc >= np.quantile(yc, 0.90)

print(f"Ba catalysts                        : {has_ba.sum()}")
print(f"mean best yield, Ba catalysts       : {yc[has_ba].mean():.2f}%")
print(f"mean best yield, everything else    : {yc[~has_ba].mean():.2f}%")
print(f"share of the lab's top decile with Ba: {100*top10[has_ba].sum()/top10.sum():.0f}%")

sc = StandardScaler().fit(Xc[tr])
m_full = lgb.LGBMRegressor(**lgb_params(42, **TUNED)).fit(sc.transform(Xc[tr]), yc[tr])
pred_full = m_full.predict(sc.transform(Xc[te]))
print(f"\nSpearman on held-out Ba catalysts   : {spearmanr(yc[te], pred_full)[0]:.3f}")

Ba catalysts                        : 291
mean best yield, Ba catalysts       : 13.76%
mean best yield, everything else    : 8.95%
share of the lab's top decile with Ba: 78%

Spearman on held-out Ba catalysts   : 0.522


### The proof that the model cannot price an unseen family

With no Ba catalysts in training, the Ba column is constant. Trees cannot split on a constant. So the
model should behave *identically* if we delete the column entirely. We test that.

In [13]:
keep = [i for i, c in enumerate(el_cols) if c != 'Ba']
sc2 = StandardScaler().fit(Xc[tr][:, keep])
m_drop = lgb.LGBMRegressor(**lgb_params(42, **TUNED)).fit(sc2.transform(Xc[tr][:, keep]), yc[tr])
pred_drop = m_drop.predict(sc2.transform(Xc[te][:, keep]))

print(f"max |difference| between the two models: {np.abs(pred_full - pred_drop).max():.10f}")
print("\n-> Exactly zero. The model prices Ba catalysts as if Ba were not there.")
print(f"   It underpredicts the best Ba catalysts by {(pred_full[yc[te] >= np.quantile(yc[te],0.9)] - yc[te][yc[te] >= np.quantile(yc[te],0.9)]).mean():.1f} yield points.")

max |difference| between the two models: 0.0000000000

-> Exactly zero. The model prices Ba catalysts as if Ba were not there.
   It underpredicts the best Ba catalysts by -9.8 yield points.


### Is Ba simply the hardest family?

No. We checked every element family with at least 50 catalysts.

In [14]:
fams = [e for e in elements if (cat_df[e].values > 0).sum() >= 50]
rows = []
for e in fams:
    h = cat_df[e].values > 0
    trf = np.where(~h)[0]; tef = np.where(h)[0]
    s = StandardScaler().fit(Xc[trf])
    mm = lgb.LGBMRegressor(**lgb_params(0, **TUNED)).fit(s.transform(Xc[trf]), yc[trf])
    rows.append((e, int(h.sum()), spearmanr(yc[tef], mm.predict(s.transform(Xc[tef])))[0],
                 100*top10[h].sum()/top10.sum()))
df_f = pd.DataFrame(rows, columns=['element','n_catalysts','spearman','pct_of_top_decile']).sort_values('spearman')
df_f = df_f.reset_index(drop=True)
ba_rank = int(df_f.index[df_f.element == 'Ba'][0]) + 1
print(f"{len(fams)} families tested. Six worst:")
print(df_f.head(6).to_string(index=False))
print(f"\nBa rank by difficulty: {ba_rank} of {len(fams)} (1 = hardest)")
print("-> Ba is not the hardest family. It is the most CONSEQUENTIAL, because it holds")
print("   most of the high-yield chemistry.")

28 families tested. Six worst:
element  n_catalysts  spearman  pct_of_top_decile
     Pd           96  0.218977           2.173913
     Cu           78  0.286157           0.000000
     Al           58  0.308192           0.000000
     Ni           82  0.328196           3.260870
     Co           71  0.499531           0.000000
     Ba          291  0.523909          78.260870

Ba rank by difficulty: 6 of 28 (1 = hardest)
-> Ba is not the hardest family. It is the most CONSEQUENTIAL, because it holds
   most of the high-yield chemistry.


## 7. Candidate catalysts

We enumerate unseen compositions using the lab's own design rules, then rank them.

In [15]:
import itertools
E = cat_df[elements].values.astype(float)
major = E.argmax(1)
SUPPORTS  = [e for e,c in pd.Series([elements[i] for i in major]).value_counts().items() if c >= 20]
PROMOTERS = [e for e,c in pd.Series([elements[j] for r,mj in zip(E,major)
             for j in np.where(r>0)[0] if j!=mj]).value_counts().items() if c >= 20]
print(f"supports the lab uses : {SUPPORTS}")
print(f"promoters (>=20 uses) : {len(PROMOTERS)} elements")

existing = set(map(tuple, np.round(Xc, 3)))
rows, labels = [], []
prep = float(cat_df['prep_enc'].mode()[0])
for s_el in SUPPORTS:
    for k,(sl,pl) in [(3,(90.0,3.3333)), (2,(93.3333,3.3333))]:
        for combo in itertools.combinations([p for p in PROMOTERS if p!=s_el], k):
            v = np.zeros(len(el_cols)); v[el_cols.index('prep_enc')] = prep
            v[el_cols.index(s_el)] = sl
            for p in combo: v[el_cols.index(p)] = pl
            if tuple(np.round(v,3)) in existing: continue
            rows.append(v); labels.append(f"{s_el}({sl:.0f}) " + " ".join(f"{p}" for p in combo))
Xcand = np.array(rows)
print(f"\nunseen candidates enumerated: {len(Xcand):,}")

sc = StandardScaler().fit(Xc)
preds = np.array([lgb.LGBMRegressor(**lgb_params(s, **TUNED)).fit(sc.transform(Xc), yc)
                  .predict(sc.transform(Xcand)) for s in range(10)])
pm = preds.mean(0)
order = np.argsort(-pm)[:10]
print("\nTop 10 by predicted best yield:")
for i in order: print(f"  {labels[i]:34s} {pm[i]:6.2f}%")
print(f"\nlab's best observed catalyst: {yc.max():.2f}%")
print("-> Predictions compress at the top. Use the RANKING, not the predicted value.")

supports the lab uses : ['Ba', 'Ti', 'La', 'Zr', 'Ca', 'Mg', 'Al', 'Si', 'Ce']
promoters (>=20 uses) : 27 elements



unseen candidates enumerated: 26,414



Top 10 by predicted best yield:
  Ba(90) Mo Zn Fe                     18.79%
  Ba(90) Mo Zn Eu                     18.57%
  Ba(93) Mo Eu                        18.56%
  Ba(90) Mo La Eu                     18.53%
  Ba(90) Mo Y La                      18.46%
  Ba(90) Mo Eu Sr                     18.38%
  Ba(90) Mo Na Eu                     18.38%
  Ba(90) Mo Fe Y                      18.36%
  Ba(90) Mo Li Eu                     18.35%
  Ba(90) W Fe Ti                      18.34%

lab's best observed catalyst: 21.50%
-> Predictions compress at the top. Use the RANKING, not the predicted value.


## 8. Provenance — which file produced each headline number

Every claim we make traces to a committed script and its stored output.

In [16]:
prov = pd.DataFrame([
 ("Row vs grouped protocol reversal",      "taniike_validation.py",     "taniike_validation.json"),
 ("Leakage mechanism (Stage-1 ablation)",  "taniike_validation.py",     "taniike_validation.json"),
 ("Quantile-normalisation ablation",       "taniike_validation.py",     "taniike_validation.json"),
 ("Grouped-CV hyperparameter retuning",    "grouped_tuning.py",         "grouped_tuning.json"),
 ("Catalyst-level reformulation",          "catalyst_level.py",         "catalyst_level.json"),
 ("Four literature-integration designs",   "phase3_lit_prior.py",       "phase3_lit_prior.json"),
 ("Ba diagnosis + drop-column proof",      "phase4_family_diagnosis.py","phase4_family_diagnosis.json"),
 ("Target audit + bootstrap intervals",    "phase5_target_audit.py",    "phase5_target_audit.json"),
 ("28-family test + learning curves",      "phase6_our_experiments.py", "phase6_our_experiments.json"),
 ("Candidate enumeration",                 "phase6_candidates.py",      "phase6_candidates.json"),
 ("Cross-preparation transfer",            "phase7_prep_ood.py",        "phase7_prep_ood.json"),
], columns=['result','script','stored output'])
print(prov.to_string(index=False))

ci = json.load(open('phase5_target_audit.json'))['E_bootstrap_ci']
print(f"\nHonest intervals on the screening metrics (bootstrap, 95%):")
print(f"  Spearman     : {ci['spearman_ci95'][0]:.3f} - {ci['spearman_ci95'][1]:.3f}")
print(f"  Enrichment   : {ci['enrichment_ci95'][0]:.2f}x - {ci['enrichment_ci95'][1]:.2f}x")
print(f"  Precision@20 : {ci['precision_at20_ci95'][0]:.2f} - {ci['precision_at20_ci95'][1]:.2f}")
print(f"\ntotal runtime: {time.time()-t0:.0f}s")

                              result                     script                stored output
    Row vs grouped protocol reversal      taniike_validation.py      taniike_validation.json
Leakage mechanism (Stage-1 ablation)      taniike_validation.py      taniike_validation.json
     Quantile-normalisation ablation      taniike_validation.py      taniike_validation.json
  Grouped-CV hyperparameter retuning          grouped_tuning.py          grouped_tuning.json
        Catalyst-level reformulation          catalyst_level.py          catalyst_level.json
 Four literature-integration designs        phase3_lit_prior.py        phase3_lit_prior.json
    Ba diagnosis + drop-column proof phase4_family_diagnosis.py phase4_family_diagnosis.json
  Target audit + bootstrap intervals     phase5_target_audit.py     phase5_target_audit.json
    28-family test + learning curves  phase6_our_experiments.py  phase6_our_experiments.json
               Candidate enumeration       phase6_candidates.py       

## Summary

**What we found.** Our original 10.6% improvement did not survive catalyst-grouped validation. The
gain came from catalyst-identity leakage, and we identified the exact code path that caused it.

**What works.** A composition-only model ranks unseen catalysts usefully — about 4x better than
random selection at finding top performers.

**What does not.** No literature-integration design beats composition alone for predicting lab
catalysts. We tested four, with criteria fixed in advance.

**What we cannot do.** The model cannot price a promoter family it has never seen. We proved this
rather than inferred it.

**Every number above was computed by running this notebook.**